# QM9 Inference Results — Molecular Properties & Dipole Reconstruction

Results from `experiments/inference/molecular` produced by `predict_from_inferred.py`.

**Section 1** — Molecular property prediction: blind vs informed across training fractions,
each with bootstrap 95% CI.  Fractions: 0.01, 0.05, 0.1, 1.0.

**Section 2** — Dipole reconstruction: `mu_inferred = ||Σ Mu_i||` vs QM9 reference
`mu` (AIMAll convention, verified MAE ≈ 0.005 D on ground-truth AIM).

In [ ]:
import os, sys, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import notebooks.inference.analyse_inference as ai
importlib.reload(ai)

EXP_DIR    = os.path.join(REPO_ROOT, 'experiments', 'inference', 'molecular')
DIPOLE_PKL = os.path.join(REPO_ROOT, 'data_curation', 'molecular', 'qm9_inferred.pkl')
FIG_DIR    = os.path.join(os.getcwd(), 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

print('EXP_DIR   :', EXP_DIR,   '→ exists:', os.path.isdir(EXP_DIR))
print('DIPOLE_PKL:', DIPOLE_PKL, '→ exists:', os.path.exists(DIPOLE_PKL))

## 1. Molecular property prediction — blind vs informed

In [ ]:
# Discover and load all prediction pkls
df = ai.load_from_experiment_dir(EXP_DIR)
fracs = ai.discover_fractions(EXP_DIR)
print(f'Loaded {len(df):,} molecules, fractions: {fracs}')
print('Pred columns:', [c for c in df.columns if '_pred_' in c][:8])

In [ ]:
# Bootstrap-CI metric table (CCC, R2, Spearman, MAE)
metric_df = ai.metric_table_inference(df, n_boot=1000, seed=0)
print(f'metric_df: {len(metric_df)} rows')

In [ ]:
# Formatted tables: CCC, R2, MAE per (variant × fraction) × property
print('=== CCC ===')
display(ai.format_metric_table(metric_df, metric='CCC', n_decimals=3))
print('\n=== R² ===')
display(ai.format_metric_table(metric_df, metric='R2',  n_decimals=3))
print('\n=== MAE ===')
display(ai.format_metric_table(metric_df, metric='MAE', n_decimals=4))

In [ ]:
import analyse_inference as ai

fig = ai.learning_curve_inference(
    metric_df, metric='R2',
    layout = '1x4', err = 'bars',
    save=os.path.join(FIG_DIR, 'learning_curve_R2.pdf'),
)
plt.show()

In [ ]:
# Parity at smallest fraction (0.01) — reveals how much AIM priors help with little data
import analyse_inference as ai
fig = ai.parity_inference(
    df, metric_df = metric_df, fraction=0.01, panel_metric='R2',
    layout = '1x4',
    save=os.path.join(FIG_DIR, 'parity_frac_0.01.pdf'),
)
plt.show()

In [ ]:
# LaTeX table for the paper (CCC with half-CI)
print(ai.latex_metric_table_inference(
    metric_df, metric='CCC', n_decimals=3,
    caption='CCC on QM9 (bootstrap 95\\% CI half-widths).',
    label='tab:inference_ccc',
))

In [ ]:
df_mu = ai.load_dipole_preds(DIPOLE_PKL)
print(f'Loaded {len(df_mu):,} molecules')
print('Columns:', list(df_mu.columns))
df_mu.head()

In [ ]:
# Bootstrap-CI metric table
dipole_metrics = ai.dipole_metric_table(df_mu, n_boot=1000, seed=0)
dipole_metrics

In [ ]:
# Hexbin parity with marginal histograms
import notebooks.inference.analyse_inference as ai
importlib.reload(ai)

fig = ai.dipole_parity(
    df_mu, metric_df = dipole_metrics, metric = 'R2',
    save=os.path.join(FIG_DIR, 'dipole_parity.pdf'),
)
plt.show()